# Label Studio + Streamlit on Google Colab via cloudflared

Notebook ini menyiapkan Label Studio dan dashboard Streamlit di Google Colab dengan `cloudflared` quick tunnel.

Environment utama yang dipakai:

- `LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED=true`
- `LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT=/content/yolo/data`
- `USE_ENFORCE_CSRF_CHECKS=false`

Port yang dipublish:

- Label Studio: `8080`
- Streamlit: `8501`

Catatan:

- Quick tunnel `trycloudflare.com` cocok untuk testing/dev.
- Kalau tunnel mati, jalankan ulang cell `Start cloudflared tunnels`.
- Folder data project diasumsikan berada di `/content/yolo/data`.


## 1. Clone repo ke Colab

Kalau repo sudah ada di `/content/yolo`, cell ini aman dijalankan lagi.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("YOLO_REPO_URL", "https://github.com/faprikaa/yolo.git")
REPO_DIR = Path("/content/yolo")

if REPO_DIR.exists():
    print(f"Repo already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/yolo


## 2. Install dependency project dan utility tunnel

In [ ]:
%cd /content/yolo
!python -m pip install --upgrade pip
!python -m pip install -r requirements-dev.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb


## 3. Siapkan folder data dan environment

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/yolo")
DATA_DIR = PROJECT_DIR / "data"
CAPTURE_DIR = DATA_DIR / "captures"
EXPORT_DIR = DATA_DIR / "exports"
DATASET_DIR = DATA_DIR / "datasets"
RUNS_DIR = DATA_DIR / "runs"

for path in (DATA_DIR, CAPTURE_DIR, EXPORT_DIR, DATASET_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.environ["LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED"] = "true"
os.environ["LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT"] = "/content/yolo/data"
os.environ["USE_ENFORCE_CSRF_CHECKS"] = "false"
os.environ["CAPTURE_DIR"] = str(CAPTURE_DIR)
os.environ["EXPORT_DIR"] = str(EXPORT_DIR)
os.environ["DATASET_DIR"] = str(DATASET_DIR)
os.environ["YOLO_RUNS_DIR"] = str(RUNS_DIR)
os.environ["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"

print("LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED=", os.environ["LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED"])
print("LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT=", os.environ["LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT"])
print("USE_ENFORCE_CSRF_CHECKS=", os.environ["USE_ENFORCE_CSRF_CHECKS"])
print("CAPTURE_DIR=", os.environ["CAPTURE_DIR"])
print("DATASET_DIR=", os.environ["DATASET_DIR"])


## 4. Jalankan Label Studio di background

Label Studio dijalankan pada `0.0.0.0:8080` supaya bisa dipublish ke tunnel.

In [ ]:
import os
import signal
import subprocess
import time
from pathlib import Path

LABEL_STUDIO_LOG = Path("/content/label-studio.log")
LABEL_STUDIO_PID = Path("/content/label-studio.pid")

if LABEL_STUDIO_PID.exists():
    try:
        old_pid = int(LABEL_STUDIO_PID.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass

command = [
    "label-studio",
    "start",
    "--host",
    "0.0.0.0",
    "--port",
    "8080",
]

with LABEL_STUDIO_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        cwd="/content/yolo",
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

LABEL_STUDIO_PID.write_text(str(process.pid), encoding="utf-8")
print(f"Label Studio PID: {process.pid}")
time.sleep(8)
!tail -n 40 /content/label-studio.log


## 5. Jalankan Streamlit di background

Dashboard Streamlit dijalankan pada `0.0.0.0:8501` supaya bisa dipublish ke tunnel juga.

In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

STREAMLIT_LOG = Path("/content/streamlit.log")
STREAMLIT_PID = Path("/content/streamlit.pid")

if STREAMLIT_PID.exists():
    try:
        old_pid = int(STREAMLIT_PID.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass

command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "app.py",
    "--server.address",
    "0.0.0.0",
    "--server.port",
    "8501",
    "--server.headless",
    "true",
]

with STREAMLIT_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        cwd="/content/yolo",
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

STREAMLIT_PID.write_text(str(process.pid), encoding="utf-8")
print(f"Streamlit PID: {process.pid}")
time.sleep(8)
!tail -n 40 /content/streamlit.log


## 6. Start cloudflared tunnels

Cell ini membuat quick tunnel terpisah untuk Label Studio dan Streamlit.

In [ ]:
import os
import re
import signal
import subprocess
import time
from pathlib import Path

def stop_existing_pid(pid_path: Path) -> None:
    if not pid_path.exists():
        return
    try:
        old_pid = int(pid_path.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass

def wait_for_public_url(log_path: Path) -> str | None:
    public_url = None
    for _ in range(30):
        time.sleep(2)
        log_text = log_path.read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log_text)
        if match:
            public_url = match.group(0)
            break
    return public_url

def start_tunnel(name: str, url: str, log_path: Path, pid_path: Path) -> str | None:
    stop_existing_pid(pid_path)
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", url],
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )
    pid_path.write_text(str(process.pid), encoding="utf-8")
    print(f"{name} cloudflared PID: {process.pid}")
    public_url = wait_for_public_url(log_path)
    if public_url:
        print(f"{name} public URL: {public_url}")
    else:
        print(f"{name} tunnel URL belum terdeteksi. Cek log: {log_path}")
        print(log_path.read_text(encoding="utf-8", errors="ignore"))
    return public_url

label_studio_url = start_tunnel(
    name="Label Studio",
    url="http://127.0.0.1:8080",
    log_path=Path("/content/cloudflared-label-studio.log"),
    pid_path=Path("/content/cloudflared-label-studio.pid"),
)

streamlit_url = start_tunnel(
    name="Streamlit",
    url="http://127.0.0.1:8501",
    log_path=Path("/content/cloudflared-streamlit.log"),
    pid_path=Path("/content/cloudflared-streamlit.pid"),
)

print("\nRingkasan URL publik:")
print("- Label Studio:", label_studio_url)
print("- Streamlit:", streamlit_url)


## 7. Cek log jika perlu

In [ ]:
!tail -n 40 /content/label-studio.log
!echo "---"
!tail -n 40 /content/streamlit.log
!echo "---"
!tail -n 40 /content/cloudflared-label-studio.log
!echo "---"
!tail -n 40 /content/cloudflared-streamlit.log


## 8. Stop service

Jalankan jika ingin mematikan Label Studio, Streamlit, dan semua tunnel `cloudflared`.

In [ ]:
import os
import signal
from pathlib import Path

for pid_file in (
    Path('/content/cloudflared-label-studio.pid'),
    Path('/content/cloudflared-streamlit.pid'),
    Path('/content/streamlit.pid'),
    Path('/content/label-studio.pid'),
):
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
            print(f"Stopped {pid_file.name}")
        except Exception as error:
            print(f"Failed to stop {pid_file.name}: {error}")
